# Основные операции PyTorch

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

In [2]:
x = torch.tensor([1, 2.0, 4])
x.grad

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
x = x.to(device)  # Тензор переедет на GPU, если устройство доступно 
x.device

device(type='cpu')

In [4]:
# включить отслеживание операций
x = torch.tensor([1.0, 2, 4], requires_grad=True)
x.requires_grad

True

In [5]:
arr = np.array([1, 2, 3.])
t = torch.from_numpy(arr)
t_copy = torch.tensor(arr)

##  Вычисление градиента

In [6]:
x = torch.tensor([2.], requires_grad=True)
y = 3*x + 1

print(y.requires_grad)
print(y.grad_fn)

True


In [7]:
y.backward()
x.grad

tensor([3.])

In [8]:
# остановка вычисления градиента
with torch.no_grad():
    print('Режим без вычисления градиента. Вычислительный граф не строится')

Режим без вычисления градиента. Вычислительный граф не строится


> корректный порядок операций на каждом шаге обучения

- optimizer.zero_grad()  # 1. Обнулили старые градиенты
- loss = compute_loss(...) # 2. Прямой проход и расчёт ошибки
- loss.backward() # 3. Обратный проход: расчет новых градиентов
- optimizer.step()  # 4. Шаг оптимизатора (обновление весов)


## Модель нейросети

In [10]:
class MLP(nn.Module):
    """Простая модель нейросети"""
    def __init__(self, in_features, hidden, num_classes):
        # Инициализация базового класса
        super().__init__()
        # Первый полносвязный слой с весами
        self.fc1 = nn.Linear(in_features, hidden)
        # Функция активации без обучаемых параметров
        self.relu = nn.ReLU()
        # Второй полносвязный слой с весами
        self.fc2 = nn.Linear(hidden, num_classes)


    def forward(self, x):
        """Определение цепочки вычислений внутри нейросети"""
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)

        return x


In [11]:
# Создание модели
model = MLP(in_features=784, hidden=128, num_classes=10)

# Перебираем все обучаемые тензоры и складываем число их элементов
total_params = sum(p.numel() for p in model.parameters())

print(total_params) # Выведет: 101770 

101770


In [12]:
# Генерируем случайный батч: 32 объекта по 784 признака в каждом
x = torch.randn(32, 784)

# Выполняем прямой проход и получаем логиты
logits = model(x)
print(logits.shape) # Выведет: torch.Size([32, 10]) 

torch.Size([32, 10])


### Оптимизатор

In [13]:
# Базовый вариант — SGD
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

# Более продвинутые адаптивные алгоритмы
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001) 